### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following: 

1) Tracking agent behaviour with logging, analytics and debugging
2) Transforming prompts, tool selection, and output formatting
3) Adding retries, fallbacks, and early termination logic
4) Applying rate limits, guardrails, and PII Detection

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C1DDC79550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C1DDC7A270>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

### Summarisation middleware

Automatically summarise conversation history when approaching token limits, preserving recent messsages while compressing older context. Summarisation is useful for the following:

1) Long running conversations that exceed context window.
2) Multi-turn dialogues with extensive history
3) Applications where preserving full conversation context matters

In [ ]:
from anyio.lowlevel import checkpoint
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Message based summarisation

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)